# DAY1_Lesson1_평가

1. 2  
2. 3  
3. 3  
4. 1  
5. 2  
6. 3  
7. 2  
8. 3  
9. 1  
10. 3

11. 토큰의 정의는 LLM이 처리하는 최소 단위인데, 조직의 기본단위가 조직구성원인 것과 유사하다.   
토큰은 한글과 영어에서 비용처리가 다르게 되는데 미국사람이 한국에서 일하면 해외거주비용을 더 줘야 하는 것 같다고 생각하면 된다  
실상은 한글은 한,글 로 구분하지만 영어는 english로 구분하는 것과 같다.


12. 입력 단가 보다 출력 토큰의 단가가 4배라는 것은 4배 비싸게 비용을 처리하고 있다는 것이다. 이럴때 출력 즉 답변에 대한 토큰 제한을 두면 비용 예산 절감에 용이하다. 이는 max_token으로 제한할 수 있다. 이렇게 제한하면 너무 긴 출력으로 인한 예산도 둘고 그만큼 들어가는 시간적 비용도 줄일 수 수 있다.

13. temperature =1.2와 max_tokens =500 파라미터를 각 0과 10 정도로 바꾼다.  
왜냐하면 temperature는 답변의 창의성을 부여하는 것으로 0일때 일관된 답변 1에 가까울 수록 창의 적인 답변을 생성하는데 이미 1을 넘었다. max_tokens는 답변 길이를 제한 할 수 있는 파라미터로 결제/배송/환불/기타 중 하나 이므로 한 단어가 나올 정로로 제한 하면 된다 500은 너무 값이 크다.

14. 후보 B의 system 메시지가 더 적합하다.  
형식이 명확하고 내용에 대한 할루시네이션을 줄일 수 있게 작성되었다.

15. 역할과 말투 사용할어휘  
역할을 부여함으로 답변에 역할에 맞는 전문적인 답을 받을 수 있고  
말투로 대화의 톤이 바뀌고  
어휘로 답변 내용에 사용되는 어휘를 조절할 수 있다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # .env 파일의 환경변수를 읽어온다.
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 읽어온다.

response = client.chat.completions.create(
    model="gpt-4o-mini",                          # 사용할 모델
    messages=[                                     # 메시지 리스트
        {"role": "user", "content": "RAG가 무엇인가요?"}
    ]
)

# 응답에서 답변만 가지고온다.
answer = response.choices[0].message.content
print("💬 답변:", answer)

In [ ]:


from dotenv import load_dotenv
load_dotenv()  # .env 파일의 환경변수를 읽어온다.
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 읽어온다.

response = client.chat.completions.create(
    model="gpt-4o-mini",                          # 사용할 모델
    messages = [
    {"role": "user", "content": "제 이름은 정현입니다."},
    {'role': 'assistant','content': '좋아요 정현님 잘 부닥드려요'},
    {'role': 'user', 'content': '내 이름이 뭐야?'}
]
)

# 답변 부분
answer = response.choices[0].message.content
print("💬 답변:", answer)


In [ ]:


def ask_llm(messages, temperature, max_tokens=500):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def classify_inquiry(text: str) -> str:
 messages = [
 {"role": "system", "content": "결제/배송/반품/기타 네 가지 중 하나로 분류해."},
 {"role": "user", "content": text},
 ]
 return ask_llm(messages, temperature=0, max_tokens=10)



test_cases = [
 "어제 주문했는데 카드 결제가 두 번 되었어요.",
 "택배가 일주일째 안 와요.",
 "사이즈가 안 맞아서 반품하고 싶어요.",
 "영업시간이 어떻게 되나요?",
]

for text in test_cases:
    category = classify_inquiry(text)
    print(f"[{category:>4}] {text}")

In [ ]:
prompt = "AI 스타트업 이름 3개를 지어주세요"

temp0 = ask_llm([{"role": "user", "content": prompt}], temperature = 0)
temp1 = ask_llm([{"role": "user", "content": prompt}], temperature =1.2)

print(f" {temp0}\n")
print(f" {temp1}")

temperature설정 0의 경우 너무 이유가 단순한 느낌임.  
1.2의 경우 선정 이유가 의미를 부여하고 0에 비해 신선함.

In [ ]:
PRICE_INPUT = 0.15   # 입력 1M 토큰당 USD
PRICE_OUTPUT = 0.60  # 출력 1M 토큰당 USD
USD_TO_KRW = 1400    # 환율

def cost_korean_won(usage):
    """몇시까지 쉬는 시간이에요?? 와...! 쉬는시간!!!"""
    input_cost = usage.prompt_tokens / 1_000_000 * PRICE_INPUT
    output_cost = usage.completion_tokens / 1_000_000 * PRICE_OUTPUT
    return (input_cost + output_cost) * USD_TO_KRW

print("📌 입력 토큰:", response.usage.prompt_tokens)
print("📌 출력 토큰:", response.usage.completion_tokens)
print("📌 총 토큰:", response.usage.total_tokens)


resp1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Python의 장점을 5가지 설명해주세요."}],
)

u = resp1.usage
print(f"비용 약 {cost_korean_won(u):.4f}원")